# 06 — Resumable VCOD single-cell runner
Run one dataset/regime/system/seed at a time. The notebook uses a stable Drive directory, resumes the latest atomic checkpoint, and delegates all model, training, and evaluation behavior to repository CLIs. Use `smoke` first, `tuning` only on validation, and `final` only after protocol approval.

In [ ]:
#@title Select exactly one experimental cell
SYSTEM = 'DS' #@param ['DS', 'VI', 'DT', 'VV', 'DM', 'VR']
DATASET = 'moca_mask' #@param ['moca_mask', 'camovid60k']
REGIME = 'default' #@param ['default', 'small_displacement', 'large_displacement']
SEED = 42 #@param {type:'integer'}
RUN_KIND = 'smoke' #@param ['smoke', 'tuning', 'final']
LEARNING_RATE = 0.0003 #@param {type:'number'}
MAX_STEPS = 10000 #@param {type:'integer'}
RUN_EVALUATION = True #@param {type:'boolean'}
FINAL_PROTOCOL_APPROVED = False #@param {type:'boolean'}
PROJECT_REPO_URL = 'https://github.com/papanag/cod-ssl.git'
PROJECT_BRANCH = 'main'
DRIVE_ROOT = '/content/drive/MyDrive/cod-ssl'

In [ ]:
# Fresh-kernel bootstrap using the same cached assets as notebooks 01–05.
from google.colab import drive
drive.mount('/content/drive')
from getpass import getpass
from pathlib import Path
import json, os, subprocess, sys, torch, yaml
project_dir = Path('/content/cod-ssl')
if (project_dir / '.git').is_dir():
    subprocess.run(['git', '-C', str(project_dir), 'fetch', 'origin', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'checkout', PROJECT_BRANCH], check=True)
    subprocess.run(['git', '-C', str(project_dir), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', PROJECT_BRANCH, PROJECT_REPO_URL, str(project_dir)], check=True)
extras = 'dev,notebooks,vcod' if SYSTEM == 'DT' else 'dev,notebooks'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f'{project_dir}[{extras}]'], check=True)
bootstrap_env = os.environ.copy()
dino_weights = Path(DRIVE_ROOT) / 'checkpoints/dinov3_vitb16.pth'
if not dino_weights.is_file():
    private_url = getpass('Private DINOv3 ViT-B/16 LVD-1689M URL: ').strip()
    if not private_url: raise ValueError('The approved DINOv3 URL is required.')
    bootstrap_env['COD_SSL_DINOV3_DOWNLOAD_URL'] = private_url
    del private_url
state_file = Path('/content/cod_ssl_bootstrap_state.json')
subprocess.run([sys.executable, str(project_dir / 'scripts/bootstrap_colab.py'),
                '--project-dir', str(project_dir), '--drive-root', DRIVE_ROOT,
                '--state-file', str(state_file)],
               cwd=project_dir, env=bootstrap_env, check=True)
bootstrap_env.pop('COD_SSL_DINOV3_DOWNLOAD_URL', None)
state = json.loads(state_file.read_text())
os.environ.update(state['environment'])
PROJECT_DIR = Path(state['project_dir'])
VCOD_ROOT = Path(state['drive_root']) / 'vcod'
MOCA_MANIFEST = VCOD_ROOT / 'manifests/moca_mask.csv'
CAMOVID_MANIFEST = VCOD_ROOT / 'manifests/camovid60k.csv'
APPROVAL_PATH = VCOD_ROOT / 'approvals/vcod_validation_approval.json'
os.environ['MOCA_MASK_MANIFEST'] = str(MOCA_MANIFEST)
os.environ['CAMOVID60K_MANIFEST'] = str(CAMOVID_MANIFEST)
os.chdir(PROJECT_DIR)
print('Ready on', state['gpu'])

In [ ]:
# Validate the requested cell and the signed dataset/checkpoint gate.
from cod_ssl.utils.run import file_sha256
if SYSTEM in {'DM', 'VR'} and RUN_KIND == 'final':
    raise ValueError('DM/VR are diagnostics and cannot be labeled final primary systems.')
if RUN_KIND == 'final' and not FINAL_PROTOCOL_APPROVED:
    raise PermissionError('Approve the validation-selected learning rate and frozen protocol before a final run.')
if DATASET == 'moca_mask' and REGIME != 'default':
    raise ValueError('MoCA-Mask uses the default regime.')
if DATASET == 'camovid60k' and REGIME == 'default':
    raise ValueError('Choose a CamoVid60K displacement regime.')
manifest = MOCA_MANIFEST if DATASET == 'moca_mask' else CAMOVID_MANIFEST
if not manifest.is_file(): raise FileNotFoundError(manifest)
if not APPROVAL_PATH.is_file():
    raise PermissionError('Run notebook 05 and complete its manual sign-off first.')
approval = json.loads(APPROVAL_PATH.read_text())
expected_hash = approval['moca_manifest_sha256' if DATASET == 'moca_mask' else 'camovid_manifest_sha256']
if file_sha256(manifest) != expected_hash:
    raise ValueError('Manifest changed after manual approval; rerun notebook 05.')
if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError('The locked run requires a BF16-capable Colab GPU.')
properties = torch.cuda.get_device_properties(0)
print({'gpu': properties.name, 'memory_gib': round(properties.total_memory / 2**30, 2),
       'system': SYSTEM, 'dataset': DATASET, 'regime': REGIME, 'seed': SEED, 'kind': RUN_KIND})

In [ ]:
# Resolve a stable, collision-resistant Drive run directory.
adapter_by_system = {'DS': 'single', 'VI': 'single', 'DT': 'gated_mamba_mix__T64_S1_target32',
                     'VV': 'vjepa_native__T64_S1_target32', 'DM': 'mean__T64_S1_target32',
                     'VR': 'vjepa_native__T64_S1_target32'}
backbone_by_system = {'DS': 'dinov3_vitb16', 'DT': 'dinov3_vitb16', 'DM': 'dinov3_vitb16',
                      'VI': 'vjepa21_vitb16', 'VV': 'vjepa21_vitb16', 'VR': 'vjepa21_vitb16'}
dataset_key = DATASET if REGIME == 'default' else f'{DATASET}_{REGIME}'
run_id = f'{dataset_key}__{SYSTEM}__{backbone_by_system[SYSTEM]}__{adapter_by_system[SYSTEM]}__seed{SEED}'
if RUN_KIND == 'final':
    RUN_DIR = VCOD_ROOT / 'runs' / run_id
elif RUN_KIND == 'tuning':
    RUN_DIR = VCOD_ROOT / 'tuning' / f'{run_id}__lr{LEARNING_RATE:g}'
else:
    RUN_DIR = VCOD_ROOT / 'smoke' / run_id
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('RUN_DIR =', RUN_DIR)

In [ ]:
# Train or resume. Periodic atomic checkpoints are written directly to the stable Drive run.
config_path = ('configs/experiments/vcod_diagnostics.yaml' if SYSTEM in {'DM', 'VR'}
               else 'configs/experiments/vcod_primary_2x2.yaml')
manifest_env = 'MOCA_MASK_MANIFEST' if DATASET == 'moca_mask' else 'CAMOVID60K_MANIFEST'
command = [sys.executable, 'scripts/train_probe.py', '--config', config_path,
           '--run-dir', str(RUN_DIR),
           f'experiment.system_id={SYSTEM}', f'experiment.seed={SEED}',
           f'dataset.name={DATASET}', f'dataset.regime={REGIME}',
           f'dataset.manifest_env={manifest_env}',
           f'training.learning_rate={LEARNING_RATE}', f'training.max_steps={MAX_STEPS}']
checkpoint = RUN_DIR / 'checkpoints/last.pt'
if checkpoint.is_file():
    command += ['--resume', str(checkpoint)]
    print('Resuming', checkpoint)
if RUN_KIND == 'smoke': command.append('--smoke')
subprocess.run(command, cwd=PROJECT_DIR, check=True)
if hasattr(os, 'sync'): os.sync()

In [ ]:
# Evaluate only the appropriate split: tuning never touches test; final never tunes on test.
if RUN_EVALUATION and RUN_KIND != 'smoke':
    split = 'val' if RUN_KIND == 'tuning' else 'test'
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--run-dir', str(RUN_DIR),
                    '--split', split, '--save-logits'], cwd=PROJECT_DIR, check=True)
    if hasattr(os, 'sync'): os.sync()
else:
    print('Evaluation skipped for this run configuration.')

In [ ]:
# Verify and display persisted artifacts.
import pandas as pd
from IPython.display import JSON, display
required = [RUN_DIR / 'config_resolved.yaml', RUN_DIR / 'environment.json',
            RUN_DIR / 'split_ids.json', RUN_DIR / 'train_log.jsonl',
            RUN_DIR / 'checkpoints.json', RUN_DIR / 'checkpoints/last.pt']
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError(f'Missing run artifacts: {missing}')
log = pd.read_json(RUN_DIR / 'train_log.jsonl', lines=True)
display(log.tail(20))
if (RUN_DIR / 'summary.json').is_file():
    display(JSON(json.loads((RUN_DIR / 'summary.json').read_text())))
print('Persisted run:', RUN_DIR)